There are a few different ways to check how well we sample a timescale in MAF. Let's compare `TgapsPercentMetric` and `GapsMetric`

**Notebook overview (added for clarity).**

This notebook compares two MAF metrics that both try to answer "how well does this cadence sample a given
timescale?", using small hand-built sequences of observation times instead of a full OpSim run, so the
behavior of each metric can be inspected in isolation:

- **`TgapsPercentMetric`**: looks at the time gaps between *consecutive* observations only, and reports
  the percentage of those gaps that fall within a chosen time window (e.g. 0.5-1.0 day). It is a
  "fraction of pairs" metric: adding more observations can dilute this percentage even if the timescale of
  interest is still being sampled just as well as before.
- **`GapsMetric`**: instead counts how many times a given timescale has been *independently* sampled
  (roughly, how many non-overlapping windows of that length contain at least one gap of the right size).
  It is a "count" metric: it only increases or stays flat as more relevant observations are added, never
  decreases.

The cells below build up several toy observation sequences (constant cadence, then mixed cadences) to show
this difference concretely: `TgapsPercentMetric` can *drop* when observations are added, while `GapsMetric`
behaves the way a science metric normally should (monotonically non-decreasing with more relevant data).

In [ ]:
import numpy as np
from rubin_sim.maf import GapsMetric, TgapsPercentMetric  # the two metrics being compared, see overview above
import matplotlib.pylab as plt

%matplotlib inline

In [ ]:
# get the dtype as expected
# MAF metrics expect a numpy structured array with named columns (as if it were a slice of an OpSim
# query result); here we only need the observation time column.
data = np.zeros(10, dtype=[("observationStartMJD", float)])

# Define a set of times
# 10 observations at exactly 1-day cadence: MJD = 0, 1, 2, ..., 9
data["observationStartMJD"] += np.arange(10)

In [ ]:
data

In [ ]:
# three views of the same 10 observations, matching how each metric "sees" the data:

# 1) raw observation times, finely binned -- just shows when each visit happened
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.25) - 0.125)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

# 2) same observation times, but binned the way GapsMetric effectively groups time into windows
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

# 3) the distribution of gaps between CONSECUTIVE observations -- this is exactly what
# TgapsPercentMetric operates on; the dashed lines mark the 0.5-1.0 day window used below
plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

In [ ]:
# a coarser view of the same observation times, for reference
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.5))
plt.xlabel("time (days)")
plt.ylabel("N obs")

In [ ]:
# TgapsPercent will tell us what percentage of consecutive observations are in the range 0.5 to 1 day
# min_time/max_time (days): the target gap window this metric reports the coverage percentage for
tgp = TgapsPercentMetric(min_time=0.5, max_time=1.0)

# Gaps will tell us how many times we have independently sampled the 0.5-1.5 day timescale.
# time_scale (hours): the characteristic timescale of interest; GapsMetric counts how many
# non-overlapping windows of roughly this length contain a suitable pair of observations
gaps = GapsMetric(time_scale=24.0)

In [ ]:
tgp.run(data)  # -> percentage of consecutive-observation gaps falling in [0.5, 1.0] days

In [ ]:
gaps.run(data)  # -> number of times the ~24h timescale has been independently sampled

In [ ]:
# what this is telling us--we are 100% optimized for observing at the 1-day timescale.
# And we have observed the timescale of interest 8 times

In [ ]:
# what happens if we double the number of observations by increasing the frequency
# now 20 observations at a constant 0.5-day cadence over the same 10-day span
data = np.zeros(20, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.arange(0, 10, 0.5)

In [ ]:
data

In [ ]:
tgp.run(data)  # every consecutive gap is now 0.5 days -> still within [0.5, 1.0], but see below

In [ ]:
gaps.run(data)  # more observations -> at least as many independent ~24h samples as before

In [ ]:
# same three diagnostic views as above, for the denser (0.5-day cadence) sequence
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.25) - 0.125)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

In [ ]:
# and increasing the frequency again
# what happens if we double the number of observations
# now 40 observations at a constant 0.25-day cadence over the same 10-day span
data = np.zeros(40, dtype=[("observationStartMJD", float)])

# Define a set of times
data["observationStartMJD"] += np.arange(0, 10, 0.25)

In [ ]:
tgp.run(data)  # consecutive gaps are now all 0.25 days -> outside the [0.5, 1.0] window -> percentage drops

In [ ]:
gaps.run(data)  # still at least as many independent ~24h samples as with fewer, sparser observations

In [ ]:
# same three diagnostic views, for the densest (0.25-day cadence) sequence
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.1) - 0.05)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

This shows one issue with `TgapsPercent`--if we have very high sampling, it (correctly) reports that we are very un-optimized in terms of observing the timescale. This is not how science metrics typically behave. Normally, we want our metrics to only increase (or stay constant) as more observations are added. This is what the `Gaps` metric does. 

In [ ]:
# Let's do 10 days at 1-day cadence, 10 days at 0.5-day cadence

# what happens if we double the number of observations
# get the dtype as expected
# 30 observations total: a MIXED-cadence sequence (unlike the constant-cadence ones above), so we can see
# how each metric responds when only part of the sequence gets denser sampling
data = np.zeros(30, dtype=[("observationStartMJD", float)])

# Define a set of times
# first 10 days: 1-day cadence (10 obs); next 10 days: 0.5-day cadence (20 obs)
data["observationStartMJD"] += np.concatenate((np.arange(0, 10, 1), np.arange(10, 20, 0.5)))

In [ ]:
data

In [ ]:
tgp.run(data)  # the 0.5-day-cadence half still keeps the consecutive-gap percentage at 100% within [0.5, 1.0]

In [ ]:
gaps.run(
    data
)  # but now MORE distinct days have been independently sampled at the ~24h timescale -> increases

In [ ]:
# same three diagnostic views, for the mixed-cadence (1-day then 0.5-day) sequence
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.1) - 0.05)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

We once again have a 100% from `TgapsPercent`, so we might think this sequence is just as good as out initial sequence of 10 days. But `Gaps` correctly notes that now we have more days independently sampled, so it increases.

In [ ]:
# Let's do 10 days at 1-day cadence, 10 days at 0.25-day cadence

# what happens if we double the number of observations
# get the dtype as expected
# 50 observations total: first half at 1-day cadence, second half at an even denser 0.25-day cadence,
# so the consecutive gaps in the second half fall OUTSIDE the TgapsPercent target window entirely
data = np.zeros(50, dtype=[("observationStartMJD", float)])

# Define a set of times
# first 10 days: 1-day cadence (10 obs); next 10 days: 0.25-day cadence (40 obs)
data["observationStartMJD"] += np.concatenate((np.arange(0, 10, 1), np.arange(10, 20, 0.25)))

In [ ]:
data

In [ ]:
tgp.run(data)  # the dense 0.25-day-cadence half drags the [0.5, 1.0]-day-gap percentage down

In [ ]:
gaps.run(data)  # roughly unchanged: still the same number of distinct days independently sampled at ~24h

In [ ]:
# same three diagnostic views, for the mixed-cadence (1-day then 0.25-day) sequence
plt.figure()
_tmp = plt.hist(data["observationStartMJD"], bins=np.arange(0, 20, 0.1) - 0.05)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Observations over time")

plt.figure()
_tmp = plt.hist(
    data["observationStartMJD"], bins=np.arange(0, 20, 0.5) - 0.25, edgecolor="black", linewidth=1.2
)
plt.xlabel("time (days)")
plt.ylabel("N obs")
plt.title("Binned like Gaps")

plt.figure()
_tmp = plt.hist(np.diff(data["observationStartMJD"]), bins=np.arange(0, 2, 0.25) - 0.125)
plt.xlabel("delta time (days)")
plt.ylabel("N")
plt.title("Like TgapsPercent")

plt.axvline(x=0.5, linestyle="--", color="k")
plt.axvline(x=1.0, linestyle="--", color="k")

Once again, adding data has caused `TgapsPercent` to drop, while `Gaps` stays nearly constant.